# RAG Lab - build, inspect and evaluate a RAG pipeline

This notebook runs **completely offline**: no API key, no Docker, no vector database.
Everything here works identically once you swap in a real LLM and a real store.

1. What can be plugged in
2. Build a pipeline and ask it something
3. Look inside a query
4. Evaluate against a golden set
5. Compare strategies and run ablations
6. Plug in a component of your own
7. Move the same pipeline to production


## 0. Setup

Point Python at the backend package. Run this notebook from the `notebooks/` folder.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "backend"))

from app.rag import RAG, Dataset, Sweep, list_components
import pandas as pd

## 1. What can be plugged in?

Every stage is a registered component, addressed by name.

In [ ]:
pd.DataFrame([
    {"kind": kind, "available": ", ".join(c["name"] for c in items)}
    for kind, items in list_components().items()
])

## 2. Build a pipeline and ask a question

`RAG("offline")` uses a hashing embedder, an in-memory store and an extractive
generator, so it needs nothing installed. Any stage can be swapped with a keyword
argument: `RAG("offline", retriever="mmr", top_n=5)`.

In [ ]:
POLICIES = [
    "The nightly hotel limit is 200 USD in standard cities and 350 USD in high-cost cities "
    "such as London, New York, Tokyo and Zurich. Stays above the limit require written "
    "approval from a vice president before booking.",
    "Hybrid employees must work from the office at least two days per week. Fully remote "
    "arrangements require VP sign-off and a 500 USD home office stipend.",
    "Passwords must be rotated every 90 days. Multi-factor authentication is mandatory for "
    "all administrative accounts and for access from outside the corporate network.",
    "The Software Engineer II salary band is 120,000 to 160,000 USD annually. "
    "Senior Engineer is 160,000 to 210,000 USD. Bands are reviewed each January.",
]

rag = RAG("offline")
rag.add_texts(POLICIES)
rag

In [ ]:
result = rag.print_answer("What is the hotel limit in high-cost cities?")

A question the corpus cannot answer should be refused, not guessed at:

In [ ]:
rag.print_answer("What is the company holiday in March?")

## 3. Look inside a query

`explain()` is the first thing to reach for when an answer looks wrong: it shows
which queries were searched, what came back with what score, and where time went.

In [ ]:
explanation = rag.explain("What is the hotel limit in high-cost cities?")

print("queries searched:", explanation["queries_searched"])
pd.DataFrame(explanation["stages"])

In [ ]:
pd.DataFrame(explanation["contexts"])[["rank", "score", "source", "preview"]]

## 4. Evaluate

A golden set is questions plus expected answers. Retrieval metrics work from
`relevant_ids`, `reference_contexts`, or - as here - by checking whether the
retrieved text contains the expected answer.

**Start with retrieval.** If the right passage is never retrieved, no amount of
prompt engineering fixes the answer.

In [ ]:
golden = Dataset.from_pairs([
    ("What is the hotel limit in high-cost cities?", "350 USD"),
    ("How many days per week must hybrid employees be in the office?", "at least two days"),
    ("How often must passwords be rotated?", "every 90 days"),
    ("What is the Senior Engineer salary band?", "160,000 to 210,000 USD"),
    ("Who approves stays above the hotel limit?", "a vice president"),
], name="policies")

pd.Series(rag.evaluate_retrieval(golden, k_values=[1, 3, 5])).to_frame("score")

Now the whole pipeline, end to end:

In [ ]:
report = rag.evaluate(golden, show_progress=False)
print(report.table())

The per-sample rows are where the useful information lives - read the failures:

In [ ]:
report.to_pandas()

In [ ]:
for sample in report.worst(2):
    print("Q:", sample.question)
    print("  got:     ", sample.answer[:140])
    print("  expected:", sample.ground_truth)
    print("  scores:  ", sample.scores)
    print()

## 5. Compare strategies

This is the loop the toolkit exists for. Each configuration re-indexes the same
corpus and is scored on the same questions, so the comparison is fair.

In [ ]:
sweep = Sweep("offline", golden, corpus=POLICIES,
              metrics=["hit_rate", "mrr", "ndcg", "context_precision", "answer_f1"])
sweep.grid(**{"retriever.type": ["dense", "bm25", "hybrid_rrf", "mmr"]})

results = sweep.run(save=False, show_progress=False)
print(results.leaderboard())

In [ ]:
results.to_pandas()

### Ablations

A grid is exponential. An ablation changes one thing at a time against a baseline:
linear, and it answers the question you usually have - *which knob matters?*

In [ ]:
ablation = Sweep("offline", golden, corpus=POLICIES,
                 metrics=["hit_rate", "ndcg", "context_precision"])
ablation.ablate(**{
    "top_n": [3, 5, 12],
    "chunker.max_tokens": [150, 350, 600],
    "reranker.type": ["identity", "keyword_overlap"],
})
print(ablation.run(save=False, show_progress=False).leaderboard())

## 6. Plug in your own component

Subclass an interface, register it, and it is immediately usable from the
constructor, YAML, the CLI, sweeps and the REST API.

In [ ]:
from app.rag import Reranker, register

@register("reranker", "nb_shortest_first")
class ShortestFirstReranker(Reranker):
    """Prefers concise passages when relevance is close - fewer tokens, less distraction."""

    def __init__(self, weight: float = 0.3):
        self._weight = weight

    async def rerank(self, query, candidates, top_n):
        longest = max((len(c.chunk.text) for c in candidates), default=1)
        for item in candidates:
            brevity = 1 - len(item.chunk.text) / longest
            item.scores["brevity"] = round(brevity, 3)
            item.score = (1 - self._weight) * item.score + self._weight * brevity
        return sorted(candidates, key=lambda c: c.score, reverse=True)[:top_n]

custom = RAG("offline", reranker={"type": "nb_shortest_first", "weight": 0.25})
custom.add_texts(POLICIES)
custom.print_answer("What is the hotel limit in high-cost cities?")

Your component competes on the leaderboard like any built-in:

In [ ]:
comparison = Sweep("offline", golden, corpus=POLICIES,
                   metrics=["hit_rate", "ndcg", "context_precision"])
comparison.grid(**{"reranker.type": ["identity", "keyword_overlap", "nb_shortest_first"]})
print(comparison.run(save=False, show_progress=False).leaderboard())

## 7. Going to production

Nothing above changes when you move to real models - only the config does:

```python
rag = RAG(
    llm={"type": "azure_openai", "deployment": "gpt-4o-mini"},
    embedder={"type": "cached", "inner": {"type": "azure_openai", "dim": 1024}},
    vectorstore={"type": "qdrant", "url": "http://localhost:6333"},
    retriever="hybrid_rrf",
    reranker={"type": "cross_encoder"},
    generator={"type": "self_rag"},
)
```

Save what you converged on and hand the file to whoever deploys it:

```python
rag.save_config("configs/my_pipeline.yaml")
```

With a real LLM you can also generate the golden set from your own corpus instead
of writing one by hand (review it before trusting it as ground truth):

```python
testset = rag.generate_testset(n_questions=100)
testset.save("golden.jsonl")
```


In [ ]:
rag.close()
custom.close()